In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random

from common_vicreg import *
import yaml
import matplotlib.colors as mcolors
import pickle

import library

In [ ]:
device = torch.device("cuda")
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = sns.color_palette()
colors

default plot setting

In [ ]:
with open("linear/config.yaml", 'r', encoding='utf-8') as f:
    configs_linear = yaml.safe_load(f)
with open("deep/config.yaml", 'r', encoding='utf-8') as f:
    configs_deep = yaml.safe_load(f)
with open("mlp/config.yaml", 'r', encoding='utf-8') as f:
    configs_mlp = yaml.safe_load(f)

config = configs_mlp
loss_type = "BarlowTwins_test_relu"

In [ ]:
def sample(batch, sizes, device, mul=0.01):
    x_out = torch.zeros((batch, 0), device=device)
    for size in sizes:
        rand = torch.rand((batch, 1), device=device).round() * 2 - 1
        x_out = torch.cat([x_out, rand.repeat(1, size)], dim=-1)
    epsilon1 = mul * torch.randn_like(x_out)
    epsilon2 = mul * torch.randn_like(x_out)
    return x_out + epsilon1, x_out + epsilon2

sample makes [+1, -1] datas

In [ ]:
def create_segment_vectors(sizes):
    total_length = sum(sizes)
    cumsum = torch.cumsum(torch.tensor(sizes), dim=-1)

    result = torch.zeros((len(sizes), total_length))

    for i in range(len(sizes)):
        start_idx = cumsum[i - 1] if i > 0 else 0
        end_idx = cumsum[i]

        result[i, start_idx:end_idx] = 1

    return result

In [ ]:
batch = config["batch"]
if "batch" in config[loss_type]:
    batch = config[loss_type]["batch"]
sizes = config["sizes"]
if "sizes" in config[loss_type]:
    sizes = config[loss_type]["sizes"]
s_in = sum(sizes)
s_out = len(sizes)

In [ ]:
sample_data = sample(batch, sizes, device, mul=0.01)
gamma = torch.bmm(sample_data[0].unsqueeze(dim=2), sample_data[1].unsqueeze(dim=1))  # b x n x n
gamma = gamma + gamma.permute(0, 2, 1)
gamma = torch.mean(gamma, dim=0).squeeze() / 2  # n x n
sns.heatmap(gamma.cpu())
plt.show()
gamma_l, gamma_v = torch.linalg.eig(gamma)
gamma_l = gamma_l.real.cpu()
gamma_v = gamma_v.real.cpu()
print(gamma_l, gamma_v)

In [ ]:
import os

if os.path.exists(f"{config["network"]}/data.npy"):
    output_dict = np.load(f"{config["network"]}/data.npy", allow_pickle=True).item()
else:
    output_dict = {}

In [ ]:
def train(loss_type, lambd=None, slope=None):
    if lambd is None:
        target_output_dict = output_dict[loss_type]
    elif slope is not None:
        if lambd not in output_dict[loss_type]:
            output_dict[loss_type][lambd] = {}
        output_dict[loss_type][lambd][slope] = {}
        target_output_dict = output_dict[loss_type][lambd][slope]
    else:
        output_dict[loss_type][lambd] = {}
        target_output_dict = output_dict[loss_type][lambd]
    set_seed(config[loss_type]["seed"])
    if config["network"] == "linear":
        encoder = nn.Linear(s_in, s_out, bias=False, device=device)
    elif config["network"] == "deep":
        mid_size = config["mid_layer_size"]
        total_layer = int(config["layer"])
        mid_layer = []
        for _ in range(total_layer - 2):
            mid_layer.append(nn.Linear(mid_size, mid_size, bias=False, device=device))
        encoder = nn.Sequential(nn.Linear(s_in, mid_size, bias=False, device=device), *mid_layer, nn.Linear(mid_size, s_out, bias=False, device=device))
    else:
        mid_size = config["mid_layer_size"]
        total_layer = int(config["layer"])
        if "layer" in config[loss_type]:
            total_layer = int(config[loss_type]["layer"])
        mid_layer = []
        if slope is not None and "leakyrelu" in loss_type:
             def activation_types():
                return nn.LeakyReLU(negative_slope=float(slope), inplace=True)
        else:
             def activation_types():
                return nn.ReLU(inplace=True)
        for _ in range(total_layer - 2):
            mid_layer.append(nn.Linear(mid_size, mid_size, bias=False, device=device))
            mid_layer.append(activation_types())
        encoder = nn.Sequential(nn.Linear(s_in, mid_size, bias=False, device=device), activation_types(), *mid_layer,
                                nn.Linear(mid_size, s_out, bias=False, device=device))
    print(encoder)
    epochs = int(config[loss_type]["epochs"])
    lr = float(config[loss_type]["lr"])
    scaling = float(config[loss_type]["scaling"])

    optimizer = torch.optim.SGD(encoder.parameters(), lr, momentum=0, weight_decay=0)

    encoder.train()

    loss_values = []
    eiglist = [[] for i in range(s_out)]
    feature_alignment = [[] for _ in range(s_out)]
    weight_singular = [[] for _ in range(s_out)]
    subspace_alignment = []
    feature_weight_r_singular_alignment = [[[] for _ in range(2)] for _ in range(2)]

    if config["network"] != "linear":
        scale_weights(encoder, scaling ** (1 / config["layer"]))
    else:
        scale_weights(encoder, scaling)

    w = get_total_weight(encoder).detach().clone()
    u0, s0, v0T = torch.linalg.svd(w)
    v0 = v0T.T

    print(f"before {s0}")
    if config["network"] == "deep" and "BarlowTwins" in loss_type and config[loss_type]["balanced"]:
        with torch.no_grad():
            layer_s_value = s0 ** (1 / config["layer"])
            before_v = None
            modules = list(filter(lambda l: isinstance(l, nn.Linear), encoder.modules()))
            for index in reversed(range(len(modules))):
                module = modules[index]
                u, s, vt = torch.linalg.svd(module.weight.detach())
                s_next = torch.zeros(u.shape[0], vt.shape[0], device=u.device)
                for i in range(s.shape[0]):
                    s_next[i, i] = s[i]
                for i in range(layer_s_value.shape[0]):
                    s_next[i, i] = layer_s_value[i]
                if before_v is not None:
                    if index == 0 and config[loss_type]["aligned"]:
                        vt.T[:, :] = gamma_v[:, :].to(device)
                    module.weight.copy_(before_v @ s_next @ vt)
                else:
                    module.weight.copy_(u @ s_next @ vt)
                before_v = vt.T.clone()
            w = get_total_weight(encoder).detach().clone()
            u0, s0, v0T = torch.linalg.svd(w)
            v0 = v0T.T

            print(f"after {s0}")
    w = get_total_weight(encoder).detach().clone()
    u0, s0, v0T = torch.linalg.svd(w)
    print(f"after {s0}")
    v0 = v0T.T

    target_output_dict["w"] = w.cpu()
    target_output_dict["u0"] = u0.cpu()
    target_output_dict["s0"] = s0.cpu()
    target_output_dict["v0"] = v0.cpu()

    for i in range(s_out):
        weight_singular[i].append(s0[i].cpu().item())
        feature_alignment[i].append(torch.norm(e[i].unsqueeze(dim=0) @ w.T).cpu().item())
    subspace_alignment.append(
        (torch.norm(v0[:, :2].T @ gamma_v[:, :2].cuda(), p='fro') ** 2 / 2).cpu().item())
    print(subspace_alignment[0])

    for i in range(2):
        for j in range(2):
            feature_weight_r_singular_alignment[i][j].append(
                ((u0[:, i] @ w @ e[j].cuda()).cpu()).item())

    eig_reversed = False
    bar = tqdm(range(1, epochs + 1))
    u_prev = None

    for epoch in bar:
        optimizer.zero_grad()
        x0, x1 = sample(batch, sizes, device, mul=0.01)
        z0, z1 = torch.chunk(encoder(torch.cat([x0, x1], dim=0)), 2, dim=0)
        c, loss = lossFn(z0, z1)
        c = c.detach()
        v = reversed(torch.linalg.eigvalsh(c))
        for i in range(len(v)):
            # print(s0[0].item()**2 * gamma_l[0].item(), v[0].item())
            eiglist[i].append(v[i].cpu().item())
        loss_values.append(loss.item())
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            bar.set_description(f'loss: {loss.item():.4f} {v[0].item():.8f}  {v[1].item():.8f}')

        w_current = get_total_weight(encoder).detach().clone()
        u_w_current, s_w_current, v_w_current_h = torch.linalg.svd(w_current)
        v_w_current = v_w_current_h.T
        if epoch > 100:
            s_diff = abs(s_w_current[0] - s_w_current[1])
            if s_diff < 0.0005 and not eig_reversed:
                eig_reversed = True
                # u_w_current, s_w_current, v_w_current = align_svd_components(u_w_current, s_w_current, v_w_current, u_prev)
                # original_score = (abs(F.cosine_similarity(u_prev[0][:, 0], u_w_current[:, 0], dim=0)) + abs(F.cosine_similarity(u_prev[0][:, 1], u_w_current[:, 1], dim=0)))
                # flipped_score =  (abs(F.cosine_similarity(u_prev[0][:, 0], u_w_current[:, 1], dim=0)) + abs(F.cosine_similarity(u_prev[0][:, 1], u_w_current[:, 0], dim=0)))
                # eig_reversed = flipped_score > original_score
                # bar.set_description(f'loss: {loss.item():.4f} {original_score.item():.8f} {flipped_score.item():.8f}')
                # if eig_reversed:
                print("reversed")
        if eig_reversed:
            u_w_current = u_w_current[:, [1, 0]]
            s_w_current = s_w_current[[1, 0]]
            v_w_current = v_w_current[:, [1, 0]]
        if u_prev is None:
            u_prev = [u_w_current.clone()]
        if epoch > 100:
            u_prev.pop(0)
            u_prev.append(u_w_current.clone())

        # Measurement

        for i in range(s_out):
            weight_singular[i].append(s_w_current[i].cpu().item())
            feature_alignment[i].append(torch.norm(e[i].unsqueeze(dim=0) @ w_current.T).cpu().item())
        subspace_alignment.append(
            (torch.norm(v_w_current[:, :2].T @ gamma_v[:, :2].cuda(), p='fro') ** 2 / 2).cpu().item())

        for i in range(2):
            for j in range(2):
                feature_weight_r_singular_alignment[i][j].append(
                    ((u_w_current[:, i] @ w_current @ e[j].cuda()).cpu()).item())

    feature_weight_r_singular_alignment = np.array(feature_weight_r_singular_alignment)
    target_output_dict["loss"] = np.array(loss_values)
    target_output_dict["eiglist"] = np.array(eiglist)
    target_output_dict["weight_singular"] = np.array(weight_singular)
    target_output_dict["subspace_alignment"] = np.array(subspace_alignment)
    target_output_dict["feature_weight_r_singular_alignment"] = feature_weight_r_singular_alignment
    target_output_dict["feature_alignment"] = np.array(feature_alignment)

In [ ]:
e = create_segment_vectors(sizes).to(device)

output_dict[loss_type] = {}
if "BarlowTwins" in loss_type:
    for lambd in config[loss_type]["lambdas"]:
        if "slope" in config[loss_type]:
            for slope in config[loss_type]["slope"]:
                lossFn = BTLoss(use_batchnorm=False, in_feature=s_out, lambd=float(lambd)).to(device)
                train(loss_type, lambd=lambd, slope=slope)
        else:
            lossFn = BTLoss(use_batchnorm="batchnorm" in loss_type, in_feature=s_out, lambd=float(lambd)).to(device)
            train(loss_type, lambd=lambd)
elif loss_type == "SimCLR":
    lossFn = SimCLRLoss(batch, 0.5).to(device)
    train(loss_type)
else:
    if "lambdas" in config[loss_type]:
        for lambd in config[loss_type]["lambdas"]:
            lossFn = VICRegLoss(cov_coeff=float(lambd)).to(device)
            train(loss_type, lambd=lambd)
    else:
        lossFn = VICRegLoss(cov_coeff=float("1")).to(device)
        train(loss_type)

In [ ]:
lambda_show = "0"
plt.plot(output_dict[loss_type][lambda_show]["eiglist"][0])
plt.plot(output_dict[loss_type][lambda_show]["eiglist"][1])
# plt.yscale('log')

In [ ]:
for i in range(2):
    plt.plot(output_dict[loss_type]["eiglist"][i])

In [ ]:
np.save(f"{config["network"]}/data.npy", output_dict)